### Challenge 1: Total Spend Per Customer

Calculate total amount spent by each customer.


In [0]:
from pyspark.sql.functions import sum as sparkSum, max as sparkMax, col, avg, year, desc, row_number
from pyspark.sql import Window

In [0]:

from pyspark.sql import Row

data = [
    Row(customer_id=1, amount=250),
    Row(customer_id=2, amount=450),
    Row(customer_id=1, amount=100),
    Row(customer_id=3, amount=300),
    Row(customer_id=2, amount=150)
]
df = spark.createDataFrame(data)
df.show()


+-----------+------+
|customer_id|amount|
+-----------+------+
|          1|   250|
|          2|   450|
|          1|   100|
|          3|   300|
|          2|   150|
+-----------+------+



In [0]:
df_with_total = df.groupBy("customer_id").agg(
    sparkSum("amount").alias("totalAmount")   
)

I grouped customer by id and found sum of amount for grouped data

In [0]:
df_with_total.show()

+-----------+-----------+
|customer_id|totalAmount|
+-----------+-----------+
|          1|        350|
|          2|        600|
|          3|        300|
+-----------+-----------+



### Challenge 2: Highest Transaction Per Day

Find the highest transaction amount for each day.


In [0]:

from pyspark.sql import Row

data = [
    Row(date='2023-01-01', amount=100),
    Row(date='2023-01-01', amount=300),
    Row(date='2023-01-02', amount=150),
    Row(date='2023-01-02', amount=200)
]
df = spark.createDataFrame(data)
df.show()


+----------+------+
|      date|amount|
+----------+------+
|2023-01-01|   100|
|2023-01-01|   300|
|2023-01-02|   150|
|2023-01-02|   200|
+----------+------+



I gouped by date and find max amount amound each group data

In [0]:
df_highest = df.groupBy("date").agg(
    sparkMax("amount")
)

In [0]:
df_highest.show()

+----------+-----------+
|      date|max(amount)|
+----------+-----------+
|2023-01-01|        300|
|2023-01-02|        200|
+----------+-----------+



### Challenge 3: Fill Missing Cities With Default

Replace null city values with 'Unknown'.


In [0]:

from pyspark.sql import Row

data = [
    Row(customer_id=1, city='Dallas'),
    Row(customer_id=2, city=None),
    Row(customer_id=3, city='Austin'),
    Row(customer_id=4, city=None)
]
df = spark.createDataFrame(data)
df.show()


+-----------+------+
|customer_id|  city|
+-----------+------+
|          1|Dallas|
|          2|  null|
|          3|Austin|
|          4|  null|
+-----------+------+



I filled unknown for city using method: fillna

In [0]:
df_filling_values = df.fillna({
  "city": "Unknown"
})

In [0]:
df_filling_values.show()

+-----------+-------+
|customer_id|   city|
+-----------+-------+
|          1| Dallas|
|          2|Unknown|
|          3| Austin|
|          4|Unknown|
+-----------+-------+



### Challenge 4: Compute Running Total by Customer

Use a window function to compute cumulative sum of purchases per customer.


In [0]:

from pyspark.sql import Row

data = [
    Row(customer_id=1, date='2023-01-01', amount=100),
    Row(customer_id=1, date='2023-01-02', amount=200),
    Row(customer_id=2, date='2023-01-01', amount=300),
    Row(customer_id=2, date='2023-01-02', amount=400)
]
df = spark.createDataFrame(data)
df.show()


+-----------+----------+------+
|customer_id|      date|amount|
+-----------+----------+------+
|          1|2023-01-01|   100|
|          1|2023-01-02|   200|
|          2|2023-01-01|   300|
|          2|2023-01-02|   400|
+-----------+----------+------+



In [0]:
window_spec = Window.partitionBy("customer_id").orderBy("date")
df_running_total = df.withColumn("Running Total", sparkSum(col("amount")).over(window_spec))

I first created window for agg function and then used it to create running total 

In [0]:
df_running_total.show()

+-----------+----------+------+-------------+
|customer_id|      date|amount|Running Total|
+-----------+----------+------+-------------+
|          1|2023-01-01|   100|          100|
|          1|2023-01-02|   200|          300|
|          2|2023-01-01|   300|          300|
|          2|2023-01-02|   400|          700|
+-----------+----------+------+-------------+



### Challenge 5: Average Sales Per Product

Find average amount per product.


In [0]:

from pyspark.sql import Row

data = [
    Row(product='A', amount=100),
    Row(product='B', amount=200),
    Row(product='A', amount=300),
    Row(product='B', amount=400)
]
df = spark.createDataFrame(data)
df.show()


+-------+------+
|product|amount|
+-------+------+
|      A|   100|
|      B|   200|
|      A|   300|
|      B|   400|
+-------+------+



In [0]:

df_avg_sales_per_customer = df.groupBy("product").agg(
  avg("amount").alias("AverageSales")
)

I grouped the data by product name and then calculated avg for each group

In [0]:
df_avg_sales_per_customer.show()

+-------+------------+
|product|AverageSales|
+-------+------------+
|      A|       200.0|
|      B|       300.0|
+-------+------------+



### Challenge 6: Extract Year From Date

Add a column to extract year from given date.


In [0]:

from pyspark.sql import Row

data = [
    Row(customer='John', transaction_date='2022-11-01'),
    Row(customer='Alice', transaction_date='2023-01-01')
]
df = spark.createDataFrame(data)
df.show()


+--------+----------------+
|customer|transaction_date|
+--------+----------------+
|    John|      2022-11-01|
|   Alice|      2023-01-01|
+--------+----------------+



In [0]:
df_with_year = df.withColumn("transaction_year", year(col("transaction_date")))

I simply added year column called transation_year using pyspark's sql function (year)

In [0]:
df_with_year.show()

+--------+----------------+----------------+
|customer|transaction_date|transaction_year|
+--------+----------------+----------------+
|    John|      2022-11-01|            2022|
|   Alice|      2023-01-01|            2023|
+--------+----------------+----------------+



### Challenge 7: Join Product and Sales Data

Join two DataFrames on product_id to get product names with amounts.


In [0]:

from pyspark.sql import Row

products = [
    Row(product_id=1, product_name='Phone'),
    Row(product_id=2, product_name='Tablet')
]
sales = [
    Row(product_id=1, amount=500),
    Row(product_id=2, amount=800),
    Row(product_id=1, amount=200)
]
df_products = spark.createDataFrame(products)
df_sales = spark.createDataFrame(sales)
df_products.show()
df_sales.show()


+----------+------------+
|product_id|product_name|
+----------+------------+
|         1|       Phone|
|         2|      Tablet|
+----------+------------+

+----------+------+
|product_id|amount|
+----------+------+
|         1|   500|
|         2|   800|
|         1|   200|
+----------+------+



In [0]:
df_product_amount = df_products\
    .join(df_sales, on= "product_id", how="right")

I using right join to join df_products and df_sales

In [0]:
df_product_amount.show()

+----------+------------+------+
|product_id|product_name|amount|
+----------+------------+------+
|         1|       Phone|   500|
|         2|      Tablet|   800|
|         1|       Phone|   200|
+----------+------------+------+



### Challenge 8: Split Tags Into Rows

Given a list of comma-separated tags, explode them into individual rows.


In [0]:

from pyspark.sql import Row

data = [
    Row(id=1, tags='tech,news'),
    Row(id=2, tags='sports,music'),
    Row(id=3, tags='food')
]
df = spark.createDataFrame(data)
df.show()


+---+------------+
| id|        tags|
+---+------------+
|  1|   tech,news|
|  2|sports,music|
|  3|        food|
+---+------------+



In [0]:
from pyspark.sql.functions import split, explode
df_split = df.withColumn("tag", split("tags", ","))
df_split.show()

+---+------------+---------------+
| id|        tags|            tag|
+---+------------+---------------+
|  1|   tech,news|   [tech, news]|
|  2|sports,music|[sports, music]|
|  3|        food|         [food]|
+---+------------+---------------+



In [0]:
df_explode = df_split.withColumn("newtags", explode("tag")).select("id", "newtags")


I first splited tags colun and stores in tag ( split() converts data into array ) 
For exampe if I split gautam phuyal(split("Gautam Phuyal", " ")) i get ["gautam", "phuyal"]
<hr>
And then I used explode function to create column of splitted data

In [0]:
df_explode.show()

+---+-------+
| id|newtags|
+---+-------+
|  1|   tech|
|  1|   news|
|  2| sports|
|  2|  music|
|  3|   food|
+---+-------+



### Challenge 9: Top-N Records Per Group

For each category, return top 2 records based on score.


In [0]:

from pyspark.sql import Row

data = [
    Row(category='A', name='x', score=80),
    Row(category='A', name='y', score=90),
    Row(category='A', name='z', score=70),
    Row(category='B', name='p', score=60),
    Row(category='B', name='q', score=85)
]
df = spark.createDataFrame(data)
df.show()


+--------+----+-----+
|category|name|score|
+--------+----+-----+
|       A|   x|   80|
|       A|   y|   90|
|       A|   z|   70|
|       B|   p|   60|
|       B|   q|   85|
+--------+----+-----+



In [0]:
window_spec = Window.partitionBy("category").orderBy(desc("score"))
df_with_rank = df.withColumn("rank", row_number().over(window_spec))


In [0]:
df_with_rank.show()

+--------+----+-----+----+
|category|name|score|rank|
+--------+----+-----+----+
|       A|   y|   90|   1|
|       A|   x|   80|   2|
|       A|   z|   70|   3|
|       B|   q|   85|   1|
|       B|   p|   60|   2|
+--------+----+-----+----+



In [0]:
df_to_2 = df_with_rank.filter(col("rank") <= 2)

Initially I defined Window by partitioning it with category and arraging in descending order by score
<br>
After that I used used row_number to each column and filter with rank <= 2

In [0]:
df_to_2.show()

+--------+----+-----+----+
|category|name|score|rank|
+--------+----+-----+----+
|       A|   y|   90|   1|
|       A|   x|   80|   2|
|       B|   q|   85|   1|
|       B|   p|   60|   2|
+--------+----+-----+----+



### Challenge 10: Null Safe Join

Join two datasets where join key might have nulls, handle using null-safe join.


In [0]:

from pyspark.sql import Row

data1 = [
    Row(id=1, name='John'),
    Row(id=None, name='Mike'),
    Row(id=2, name='Alice')
]
data2 = [
    Row(id=1, salary=5000),
    Row(id=None, salary=3000)
]
df1 = spark.createDataFrame(data1)
df2 = spark.createDataFrame(data2)
df1.show()
df2.show()


+----+-----+
|  id| name|
+----+-----+
|   1| John|
|null| Mike|
|   2|Alice|
+----+-----+

+----+------+
|  id|salary|
+----+------+
|   1|  5000|
|null|  3000|
+----+------+



In [0]:
joined_df = df1.join(
    df2,
    df1["id"].eqNullSafe(df2["id"]),
    "inner"
)

null-safe operator (<=> )can be used in sql but i found eqNullSafe() can be used in pyspark and i safely joned two tables

In [0]:
joined_df.show()

+----+----+----+------+
|  id|name|  id|salary|
+----+----+----+------+
|null|Mike|null|  3000|
|   1|John|   1|  5000|
+----+----+----+------+

